# Frozen confirmation study
Standardized choices only. Fit additional teacher/controller seeds on TRAIN before evaluating the frozen 128-case test. No test-based tuning. Run on T4; upload the minimal bundle from `confirmation_package`. No credentials are uploaded. Download all outputs before releasing the runtime.


In [ ]:
import sys,subprocess,torch,json
print(json.dumps({'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__}))
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==5.15.1','peft==0.18.1'])


In [ ]:
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
from IPython.display import clear_output
bundle=Path('/content/confirmation-study.zip')
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='e4dbcb9e9ce928cbc2ce1405a2533075710c3aac0f9749cc18e2fc246652e66f'
root=Path('/content/confirmation')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(bundle) as z:
    assert all((root/n).resolve().is_relative_to(root.resolve()) for n in z.namelist())
    z.extractall(root)
env=os.environ.copy()
env.update(SP_LENSE_REPO=str(root),PYTHONPATH=str(root/'src'))
work=root/'work'
work.mkdir()
jobs=[('fit','m08',43),('fit','m08',44),('fit','m2',42),('evaluate','m08',42),('evaluate','m08',43),('evaluate','m08',44),('evaluate','m2',42)]
receipts=[]
began=time.monotonic()
for mode,model,seed in jobs:
    if time.monotonic()-began>=10800:
        receipts.append({'mode':mode,'model':model,'seed':seed,'state':'not_run_total_time_limit'})
        break
    if mode=='evaluate' and (model,seed)!=('m08',42):
        fit=work/f'{model}_s{seed}'/'fit'
        if not (fit/'TRAINING.json').exists() or (fit/'FAILURE.json').exists():
            receipts.append({'mode':mode,'model':model,'seed':seed,'state':'not_run_fit_failed'})
            continue
    out=work/f'{model}_s{seed}'/mode
    logpath=work/f'{model}_s{seed}_{mode}.log'
    started=time.monotonic()
    with logpath.open('w') as log:
        proc=subprocess.Popen([sys.executable,'-m','sp_lense.research2.confirmation',mode,str(root),str(out),model,str(seed)],env=env,stdout=log,stderr=subprocess.STDOUT)
        try:
            while proc.poll() is None:
                if time.monotonic()-started>3600 or time.monotonic()-began>10800:
                    proc.kill();proc.wait()
                    break
                clear_output(wait=True)
                print(f'{mode} {model} seed {seed}; {len(receipts)}/{len(jobs)} jobs finished')
                status=out/'STATUS.json'
                print(status.read_text() if status.exists() else 'Starting')
                print('Completed jobs:',json.dumps(receipts))
                time.sleep(5)
        finally:
            if proc.poll() is None:proc.kill();proc.wait()
    receipt={'mode':mode,'model':model,'seed':seed,'exit_code':proc.returncode,'seconds':time.monotonic()-started}
    receipts.append(receipt)
    (work/'JOBS.json').write_text(json.dumps(receipts,indent=2))
    # Preserve partial results after every job; no overwrite of native run records.
    shutil.make_archive('/content/confirmation-results','zip',work)
    print(receipt)
    if proc.returncode: print(logpath.read_text()[-3000:])
print('ALL_PLANNED_JOBS_ACCOUNTED_FOR')
print(json.dumps(receipts,indent=2))
shutil.make_archive('/content/confirmation-results','zip',work)


In [ ]:
from google.colab import files
print(hashlib.sha256(Path('/content/confirmation-results.zip').read_bytes()).hexdigest())
files.download('/content/confirmation-results.zip')
